In [1]:
import csv
import os
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://localhost:3000"
HEADERS = {"User-Agent": "MineriaWeb-2026-2/1.0 (+scraper-tienda-virtual)"}


def get_soup(url: str, parser: str = "html.parser") -> BeautifulSoup:
    respuesta = requests.get(url, headers=HEADERS, timeout=10)
    respuesta.raise_for_status()
    return BeautifulSoup(respuesta.content, parser)

In [10]:
sitemap_index = get_soup(f"{BASE_URL}/sitemap.xml", "xml")
sub_sitemaps = [loc.text for loc in sitemap_index.find_all("loc")]
ordenes_sitemap_url = next(url for url in sub_sitemaps if "sitemap-ordenes" in url)

sitemap_ordenes = get_soup(ordenes_sitemap_url, "xml")
urls_ordenes_sitemap = [loc.text for loc in sitemap_ordenes.find_all("loc")]
print("URLs declaradas en sitemap-ordenes.xml:", urls_ordenes_sitemap)

ordenes_base_url = next(url for url in urls_ordenes_sitemap if url.rstrip("/").endswith("/ordenes"))
print("\nListado base de ordenes:", ordenes_base_url)

ordenes_urls = urls_ordenes_sitemap[1:]
print("\nListado de ordenes:", ordenes_urls)

URLs declaradas en sitemap-ordenes.xml: ['http://localhost:3000/ordenes', 'http://localhost:3000/ordenes/300', 'http://localhost:3000/ordenes/299', 'http://localhost:3000/ordenes/298', 'http://localhost:3000/ordenes/297', 'http://localhost:3000/ordenes/296', 'http://localhost:3000/ordenes/295', 'http://localhost:3000/ordenes/294', 'http://localhost:3000/ordenes/293', 'http://localhost:3000/ordenes/292', 'http://localhost:3000/ordenes/291', 'http://localhost:3000/ordenes/290', 'http://localhost:3000/ordenes/289', 'http://localhost:3000/ordenes/288', 'http://localhost:3000/ordenes/287', 'http://localhost:3000/ordenes/286', 'http://localhost:3000/ordenes/285', 'http://localhost:3000/ordenes/284', 'http://localhost:3000/ordenes/283', 'http://localhost:3000/ordenes/282', 'http://localhost:3000/ordenes/281', 'http://localhost:3000/ordenes/280', 'http://localhost:3000/ordenes/279', 'http://localhost:3000/ordenes/278', 'http://localhost:3000/ordenes/277', 'http://localhost:3000/ordenes/276', '

In [35]:
def scrape_section_orden(section) -> dict:
    return {
        "cliente-id": section.get("data-cliente-id"),
        "orden-id": int(section.get("data-orden-id")),
        "numero": section.get("data-numero"),
        "fecha": section.get("data-fecha"),
        "sub-total": int(section.get("data-sub-total")),
        "total": section.get("data-total")
    }

In [36]:
ordenes = []

for url in ordenes_urls[:3]:
    soup = get_soup(url)
    section = soup.select_one("section[data-cliente-id]")
    if not section:
        break
    orden = scrape_section_orden(section)
    ordenes.append(orden)
print(f"Ordenes {ordenes}")

Ordenes [{'cliente-id': '110', 'orden-id': 300, 'numero': 'OR-00300', 'fecha': '2026-08-10', 'sub-total': 59586, 'total': '70311.48'}, {'cliente-id': '63', 'orden-id': 299, 'numero': 'OR-00299', 'fecha': '2026-08-07', 'sub-total': 17240, 'total': '20343.2'}, {'cliente-id': '42', 'orden-id': 298, 'numero': 'OR-00298', 'fecha': '2026-08-04', 'sub-total': 9091, 'total': '10727.38'}]


In [ ]:
## TODO: import clientes.csv to extract name, last name, etc.